# memory
> 

In [ ]:
#| default_exp memory

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from pyld import jsonld
import httpx
from rdflib import Graph, Dataset, URIRef, Literal, Namespace
from rdflib.namespace import RDF, RDFS
from datetime import datetime
from fastcore.basics import patch
from claudette import Chat, toolloop, models

## Semantic Memory Module

In [ ]:
#| export
class SemanticMemory:
    def __init__(self):
        # Existing initialization
        self.dataset = Dataset()
        self.default_graph = self.dataset.graph(URIRef("urn:x-arq:DefaultGraph"))
        self.contexts = {}
        self.indices = {
            "by_id": {},
            "by_type": {},
            "by_index": {},
            "by_label": {},
            "by_description": {}
        }
        self.uri_cache = {}
        self.client = httpx.Client(follow_redirects=True)
        
        # New context registry
        self.context_registry = {
            # "context_id": {
            #     "context": {...},  # The actual context object
            #     "source": "...",   # Where this context came from (URI or generated ID)
            #     "parent": "...",   # Parent context ID if this is a scoped context
            #     "scoped_contexts": {},  # Map of term -> context_id for terms with scoped contexts
            #     "used_by": set()   # Set of entity IDs that use this context
            # }
        }
        
        # Map to track which entities use which contexts
        self.entity_contexts = {
            # "entity_id": {
            #     "primary_context": "context_id",
            #     "property_contexts": {
            #         "property1": "context_id",
            #         "property2": "context_id"
            #     }
            # }
        }
        
        # Storage for original (non-expanded) JSON-LD
        self.original_data = {
            # "entity_id": {...}  # Original JSON-LD data
        }


In [ ]:
#| export
@patch
def fetch_context(self:SemanticMemory, context_uri):
    """Fetch and cache a JSON-LD context"""
    if context_uri in self.contexts:
        return self.contexts[context_uri]["document"]
        
    # For simplicity in our initial implementation, we'll handle local contexts
    # and leave remote fetching for later
    if isinstance(context_uri, dict):
        # It's already a context object
        self.contexts[str(id(context_uri))] = {
            "document": context_uri,
            "last_fetched": datetime.now().isoformat(),
            "dependencies": []
        }
        return context_uri
    
    # For now, just return a simple context if URI not found
    # We'll implement proper HTTP fetching in a later step
    simple_context = {}
    self.contexts[context_uri] = {
        "document": simple_context,
        "last_fetched": datetime.now().isoformat(),
        "dependencies": []
    }
    return simple_context


In [ ]:
#| export
def follow_your_nose(self, uri):
    """Retrieve and integrate linked data following your nose principle"""
    # Check if we already have comprehensive data about this URI
    if uri in self.indices["by_id"] and len(self.indices["by_id"][uri]) > 3:
        return self.indices["by_id"][uri]
    
    # Attempt to dereference the URI
    try:
        response = self.client.get(uri)
        if response.status_code == 200:
            content_type = response.headers.get("Content-Type", "")
            
            if "application/ld+json" in content_type or "application/json" in content_type:
                # Process JSON-LD data
                data = response.json()
                return self.add_jsonld(data)
            elif "text/turtle" in content_type:
                # Process Turtle data
                g = Graph()
                g.parse(data=response.text, format="turtle")
                
                # Convert to JSON-LD and add to our system
                jsonld_data = json.loads(g.serialize(format="json-ld"))
                return self.add_jsonld(jsonld_data)
    except Exception as e:
        print(f"Error following nose to {uri}: {e}")
    
    return None


In [ ]:
#| export
@patch
def _register_contexts(self:SemanticMemory, data, parent_context_id=None):
    """Register all contexts in a JSON-LD document, including scoped contexts"""
    if not isinstance(data, dict):
        return None
    
    # Handle the main context
    if "@context" in data:
        context = data["@context"]
        context_id = self._get_context_id(context)
        
        # If this context is already registered, just return its ID
        if context_id in self.context_registry:
            # Update parent relationship if this is a scoped context
            if parent_context_id:
                self.context_registry[context_id]["parent"] = parent_context_id
            return context_id
        
        # Register the new context
        self.context_registry[context_id] = {
            "context": context,
            "source": "document",  # Or could be URI if it's a remote context
            "parent": parent_context_id,
            "scoped_contexts": {},
            "used_by": set()
        }
        
        # Look for scoped contexts within this context
        if isinstance(context, dict):
            for term, value in context.items():
                if isinstance(value, dict) and "@context" in value:
                    # This term has a scoped context
                    scoped_context_id = self._register_contexts(value, context_id)
                    self.context_registry[context_id]["scoped_contexts"][term] = scoped_context_id
        
        return context_id
    
    return None

In [ ]:
#| export
@patch
def _get_context_id(self:SemanticMemory, context):
    """Generate a stable ID for a context"""
    # If context is a URI string, use it directly
    if isinstance(context, str):
        return context
    
    # If it's a list, concatenate the IDs of each component
    if isinstance(context, list):
        return "context:" + "_".join(self._get_context_id(c) for c in context)
    
    # For a context object, use a hash of its JSON representation
    if isinstance(context, dict):
        context_str = json.dumps(context, sort_keys=True)
        import hashlib
        return "context:" + hashlib.md5(context_str.encode()).hexdigest()
    
    # Fallback
    return f"context:{id(context)}"


In [ ]:
#| export
@patch
def add_jsonld(self:SemanticMemory, data, context=None):
    """Add JSON-LD data while preserving context information"""
    import uuid
    
    # If context is provided, merge it with the data
    if context:
        if isinstance(data, dict):
            data = data.copy()
            if "@context" in data:
                # Merge contexts
                if isinstance(data["@context"], list):
                    if isinstance(context, list):
                        data["@context"] = context + data["@context"]
                    else:
                        data["@context"] = [context] + data["@context"]
                else:
                    if isinstance(context, list):
                        data["@context"] = context + [data["@context"]]
                    else:
                        data["@context"] = [context, data["@context"]]
            else:
                data["@context"] = context
    
    # Process and register contexts
    if isinstance(data, dict) and "@context" in data:
        # Extract all contexts (including scoped contexts)
        self._register_contexts(data)
    
    # Store the original data
    entity_id = None
    if isinstance(data, dict) and "@id" in data:
        entity_id = data["@id"]
    else:
        # Generate an ID if none exists
        entity_id = f"urn:uuid:{uuid.uuid4()}"
        if isinstance(data, dict):
            data = data.copy()
            data["@id"] = entity_id
    
    # Store the original data
    self.original_data[entity_id] = data
    
    # Expand the data for our indices
    expanded = jsonld.expand(data)
    
    # Update our indices based on the expanded data
    self._update_indices(expanded)
    self._update_indices_with_labels(expanded)
    
    # Convert to RDF and add to our dataset
    g = Graph()
    g.parse(data=json.dumps(expanded), format='json-ld')
    self.default_graph += g
    
    # Track which context(s) this entity uses
    if entity_id not in self.entity_contexts:
        self.entity_contexts[entity_id] = {
            "primary_context": None,
            "property_contexts": {}
        }
    
    # If the entity has a context, record it as the primary context
    if isinstance(data, dict) and "@context" in data:
        context_id = self._get_context_id(data["@context"])
        self.entity_contexts[entity_id]["primary_context"] = context_id
        self.context_registry[context_id]["used_by"].add(entity_id)
    
    return expanded


In [ ]:
#| export
@patch
def _update_indices(self:SemanticMemory, expanded_data):
    """Update our custom indices based on expanded JSON-LD data"""
    if not isinstance(expanded_data, list):
        return
        
    for node in expanded_data:
        if not isinstance(node, dict):
            continue
            
        # Extract the node ID if available
        node_id = node.get('@id')
        if node_id:
            self.indices["by_id"][node_id] = node
            
        # Extract types for type index
        types = node.get('@type', [])
        if not isinstance(types, list):
            types = [types]
            
        for type_uri in types:
            if type_uri not in self.indices["by_type"]:
                self.indices["by_type"][type_uri] = []
            self.indices["by_type"][type_uri].append(node)


In [ ]:
#| export
@patch
def query_by_id(self:SemanticMemory, entity_id):
    """Retrieve an entity by its ID"""
    # Check our JSON-LD index first (faster)
    if entity_id in self.indices["by_id"]:
        return self.indices["by_id"][entity_id]
    
    # Fall back to RDF query
    entity_uri = URIRef(entity_id)
    results = {}
    
    for s, p, o in self.default_graph.triples((entity_uri, None, None)):
        pred = str(p)
        if pred not in results:
            results[pred] = []
        
        if isinstance(o, URIRef):
            results[pred].append({"@id": str(o)})
        elif isinstance(o, Literal):
            results[pred].append({"@value": str(o)})
    
    return results if results else None


In [ ]:
#| export
@patch
def query_by_type(self:SemanticMemory, type_uri):
    """Retrieve all entities of a specific type"""
    # Check our type index first
    if type_uri in self.indices["by_type"]:
        return self.indices["by_type"][type_uri]
    
    # Fall back to RDF query
    type_uri_ref = URIRef(type_uri)
    entities = []
    
    # Find all subjects with the specified type
    for s in self.default_graph.subjects(RDF.type, type_uri_ref):
        # Get all properties for this subject
        entity = {"@id": str(s), "@type": type_uri}
        
        for p, o in self.default_graph.predicate_objects(s):
            pred = str(p)
            if pred == str(RDF.type):
                continue  # Already handled
                
            if pred not in entity:
                entity[pred] = []
                
            if isinstance(o, URIRef):
                entity[pred].append({"@id": str(o)})
            elif isinstance(o, Literal):
                entity[pred].append({"@value": str(o)})
        
        entities.append(entity)
    
    return entities


In [ ]:
#| export
@patch
def _update_indices_with_labels(self:SemanticMemory, expanded_data):
    """Update indices with focus on labels and descriptions rather than URIs"""
    # Label properties to look for
    label_properties = [
        "http://www.w3.org/2000/01/rdf-schema#label",
        "http://www.w3.org/2004/02/skos/core#prefLabel",
        "http://schema.org/name",
        "http://purl.org/dc/terms/title"
    ]
    
    # Description properties
    description_properties = [
        "http://www.w3.org/2000/01/rdf-schema#comment",
        "http://purl.org/dc/terms/description",
        "http://schema.org/description"
    ]
    
    # Create label and description indices
    if "by_label" not in self.indices:
        self.indices["by_label"] = {}
    
    if "by_description" not in self.indices:
        self.indices["by_description"] = {}
    
    # Process expanded data
    for node in expanded_data if isinstance(expanded_data, list) else [expanded_data]:
        if not isinstance(node, dict):
            continue
            
        node_id = node.get('@id')
        if not node_id:
            continue
            
        # Extract labels
        for label_prop in label_properties:
            if label_prop in node:
                for value_obj in node[label_prop]:
                    if isinstance(value_obj, dict) and '@value' in value_obj:
                        label = value_obj['@value']
                        # Index by label
                        if label not in self.indices["by_label"]:
                            self.indices["by_label"][label] = []
                        if node not in self.indices["by_label"][label]:
                            self.indices["by_label"][label].append(node)
        
        # Extract descriptions
        for desc_prop in description_properties:
            if desc_prop in node:
                for value_obj in node[desc_prop]:
                    if isinstance(value_obj, dict) and '@value' in value_obj:
                        desc = value_obj['@value']
                        # Index by description keywords
                        keywords = [w.lower() for w in desc.split() if len(w) > 3]
                        for keyword in keywords:
                            if keyword not in self.indices["by_description"]:
                                self.indices["by_description"][keyword] = []
                            if node not in self.indices["by_description"][keyword]:
                                self.indices["by_description"][keyword].append(node)


In [ ]:
# Run our test
def test_semantic_memory_basics():
    memory = SemanticMemory()
    
    # Simple JSON-LD document to test with
    test_data = {
        "@context": {
            "name": "http://schema.org/name",
            "knows": {
                "@id": "http://schema.org/knows",
                "@type": "@id"
            }
        },
        "@id": "http://example.org/person/alice",
        "@type": "http://schema.org/Person",
        "name": "Alice",
        "knows": "http://example.org/person/bob"
    }
    
    # Add to memory
    expanded = memory.add_jsonld(test_data)
    print("Expanded data:", json.dumps(expanded, indent=2)[:100] + "...")
    
    # Test retrieval by ID
    alice = memory.query_by_id("http://example.org/person/alice")
    
    # Print results to verify
    print("Retrieved by ID:", alice is not None)
    if alice:
        print(f"ID: {alice.get('@id')}")
        name_values = alice.get('http://schema.org/name', [])
        if name_values:
            print(f"Name: {name_values[0].get('@value')}")
    
    # Test retrieval by type
    people = memory.query_by_type("http://schema.org/Person")
    print(f"Found {len(people)} people")
    
    return memory

In [ ]:
test_semantic_memory_basics()

Expanded data: [
  {
    "@id": "http://example.org/person/alice",
    "@type": [
      "http://schema.org/Person"
...
Retrieved by ID: True
ID: http://example.org/person/alice
Name: Alice
Found 1 people


<__main__.SemanticMemory>

In [ ]:
#| export
@patch
def create_entity_with_uuid(self:SemanticMemory, data, type_uri=None):
    """Create a new entity with a UUID identifier"""
    import uuid
    
    # Generate a UUID
    entity_uuid = str(uuid.uuid4())
    entity_id = f"urn:uuid:{entity_uuid}"
    
    # Create entity with the UUID as ID
    entity = {
        "@id": entity_id
    }
    
    # Add type if provided
    if type_uri:
        entity["@type"] = type_uri
    
    # Add any additional data
    if isinstance(data, dict):
        for key, value in data.items():
            if key not in ["@id", "@context"]:
                entity[key] = value
    
    # Add to memory
    self.add_jsonld(entity)
    
    return entity_id


In [ ]:
#| export
@patch
def resolve_did(self:SemanticMemory, did):
    """Resolve a DID to its DID Document"""
    # Check if we already have this DID in our memory
    if did in self.indices["by_id"]:
        return self.indices["by_id"][did]
    
    # In a real implementation, we would use a DID resolver library
    # For now, we'll just simulate it with a basic structure
    
    # Example DID document structure
    did_document = {
        "@context": "https://www.w3.org/ns/did/v1",
        "id": did,
        "authentication": [
            {
                "id": f"{did}#keys-1",
                "type": "Ed25519VerificationKey2018",
                "controller": did,
                "publicKeyBase58": "simulated-public-key"
            }
        ]
    }
    
    # Add to our memory
    self.add_jsonld(did_document)
    
    return did_document


In [ ]:
#export
@patch
def dereference_uri(self:SemanticMemory, uri, force_refresh=False, ttl_seconds=3600):
    """
    Dereference a URI and cache the result
    
    Args:
        uri: The URI to dereference
        force_refresh: Whether to force a refresh even if cached
        ttl_seconds: Time-to-live in seconds for cache entries
    
    Returns:
        The dereferenced data
    """
    from datetime import timedelta  # Import timedelta directly
    now = datetime.now()
    
    # Check cache first (unless forced refresh)
    if not force_refresh and uri in self.uri_cache:
        cache_entry = self.uri_cache[uri]
        
        # Check if entry is still valid
        if cache_entry["expires_at"] > now:
            return cache_entry["data"]
    
    # Need to fetch the resource
    try:
        # Prepare headers for content negotiation
        headers = {
            "Accept": "application/ld+json, application/json;q=0.9, text/turtle;q=0.8"
        }
        
        response = self.client.get(uri, headers=headers)
        
        if response.status_code == 200:
            content_type = response.headers.get("Content-Type", "").split(";")[0].strip()
            
            # Process based on content type
            if "application/ld+json" in content_type or "application/json" in content_type:
                data = response.json()
                format = "json-ld"
            elif "text/turtle" in content_type:
                g = Graph()
                g.parse(data=response.text, format="turtle")
                data = json.loads(g.serialize(format="json-ld"))
                format = "turtle"
            else:
                # Default fallback
                data = {"@id": uri, "http://schema.org/url": uri}
                format = "unknown"
            
            # Cache the result
            self.uri_cache[uri] = {
                "data": data,
                "format": format,
                "fetched_at": now,
                "expires_at": now + timedelta(seconds=ttl_seconds),  # Fixed here
                "status": response.status_code
            }
            
            # Also add to our memory store
            self.add_jsonld(data)
            
            return data
        else:
            # Cache the failure too, but with shorter TTL
            self.uri_cache[uri] = {
                "data": {"@id": uri, "error": f"HTTP {response.status_code}"},
                "format": "error",
                "fetched_at": now,
                "expires_at": now + timedelta(seconds=min(ttl_seconds, 300)),  # Fixed here
                "status": response.status_code
            }
            return self.uri_cache[uri]["data"]
            
    except Exception as e:
        # Cache the error too
        self.uri_cache[uri] = {
            "data": {"@id": uri, "error": str(e)},
            "format": "error",
            "fetched_at": now,
            "expires_at": now + timedelta(seconds=min(ttl_seconds, 300)),  # Fixed here
            "status": 0
        }
        return self.uri_cache[uri]["data"]


In [ ]:
#| export
@patch
def follow_link(self:SemanticMemory, uri, predicate=None, limit=5):
    """
    Follow links from a resource to related resources
    
    Args:
        uri: Starting URI
        predicate: Optional predicate to follow (if None, follows all links)
        limit: Maximum number of links to follow
    
    Returns:
        Dictionary of linked resources
    """
    # First, ensure we have data about the starting URI
    start_data = self.dereference_uri(uri)
    
    linked_resources = {}
    links_followed = 0
    
    # Get expanded data to normalize predicates
    expanded = jsonld.expand(start_data)
    
    for node in expanded:
        if node.get('@id') != uri:
            continue
            
        # Iterate through all predicates in the node
        for pred, values in node.items():
            if pred == '@id' or pred == '@type':
                continue
                
            # If a specific predicate was requested, skip others
            if predicate and pred != predicate:
                continue
                
            for value in values:
                if isinstance(value, dict) and '@id' in value:
                    target_uri = value['@id']
                    
                    # Only follow HTTP URIs
                    if target_uri.startswith('http') and links_followed < limit:
                        # Dereference the linked resource
                        linked_data = self.dereference_uri(target_uri)
                        linked_resources[target_uri] = linked_data
                        links_followed += 1
    
    return linked_resources


## Manage Context and Containers

In [ ]:
#| export
@patch
def manage_context(self:SemanticMemory, context_uri, force_refresh=False, ttl_seconds=86400):
    """
    Fetch, cache, and process a JSON-LD context
    
    Args:
        context_uri: URI of the context to manage
        force_refresh: Whether to force a refresh even if cached
        ttl_seconds: Time-to-live in seconds for context cache entries (default: 1 day)
    
    Returns:
        The processed context
    """
    from datetime import timedelta  # Import timedelta directly
    now = datetime.now()
    
    # Handle context objects directly
    if isinstance(context_uri, dict):
        context_id = str(id(context_uri))
        self.contexts[context_id] = {
            "document": context_uri,
            "fetched_at": now,
            "expires_at": now + timedelta(seconds=ttl_seconds),  # Fixed here
            "dependencies": []
        }
        return context_uri
    
    # Check if we already have this context cached
    if not force_refresh and context_uri in self.contexts:
        context_entry = self.contexts[context_uri]
        
        # Check if entry is still valid
        if context_entry["expires_at"] > now:
            return context_entry["document"]
    
    # Need to fetch the context
    try:
        # Use our existing dereference method
        context_data = self.dereference_uri(context_uri, force_refresh, ttl_seconds)
        
        # Extract the actual context
        if "@context" in context_data:
            context = context_data["@context"]
        else:
            context = context_data
        
        # Process context dependencies (@import)
        dependencies = []
        if isinstance(context, dict) and "@import" in context:
            import_uri = context["@import"]
            imported_context = self.manage_context(import_uri, force_refresh, ttl_seconds)
            dependencies.append(import_uri)
            
            # Merge imported context with current one
            # Note: This is simplified - a real implementation would handle merging properly
            if isinstance(imported_context, dict):
                for key, value in imported_context.items():
                    if key not in context:
                        context[key] = value
        
        # Store in our context registry
        self.contexts[context_uri] = {
            "document": context,
            "fetched_at": now,
            "expires_at": now + timedelta(seconds=ttl_seconds),  # Fixed here
            "dependencies": dependencies
        }
        
        return context
    
    except Exception as e:
        # Cache the error too, but with shorter TTL
        self.contexts[context_uri] = {
            "document": {"error": str(e)},
            "fetched_at": now,
            "expires_at": now + timedelta(seconds=min(ttl_seconds, 300)),  # Fixed here
            "dependencies": []
        }
        return self.contexts[context_uri]["document"]


In [ ]:
#| export
@patch
def build_type_container(self:SemanticMemory, base_type=None, include_subtypes=True):
    """
    Build a JSON-LD 1.1 @type container for efficient type-based access
    
    Args:
        base_type: Optional base type to filter by (if None, includes all types)
        include_subtypes: Whether to include subtypes of the base_type
    
    Returns:
        A JSON-LD document with @container: @type structure
    """
    # Create the container structure
    container = {
        "@context": {
            "@version": 1.1,
            "resources": {
                "@id": "http://memory.org/resources",
                "@container": "@type"
            }
        },
        "resources": {}
    }
    
    # If we're looking for a specific type and its subtypes, we need to find all subtypes
    subtypes = set()
    if base_type and include_subtypes:
        # Find all rdfs:subClassOf relationships
        for s, p, o in self.default_graph.triples((None, RDFS.subClassOf, URIRef(base_type))):
            subtypes.add(str(s))
    
    # Get all entities from our index
    for type_uri, entities in self.indices["by_type"].items():
        # Skip if we're filtering by type and this doesn't match
        if base_type and type_uri != base_type and type_uri not in subtypes:
            continue
            
        # Add each entity to the appropriate type container
        for entity in entities:
            entity_id = entity.get('@id')
            if not entity_id:
                continue
                
            # Create the type entry if it doesn't exist
            if type_uri not in container["resources"]:
                container["resources"][type_uri] = []
                
            # Add a reference to this entity
            container["resources"][type_uri].append({"@id": entity_id})
    
    return container


In [ ]:
#| export
@patch
def build_property_container(self:SemanticMemory, property_uri):
    """
    Build a JSON-LD 1.1 @index container for property-based access
    
    Args:
        property_uri: The property to index by
    
    Returns:
        A JSON-LD document with @container: @index structure
    """
    # Create the container structure
    container = {
        "@context": {
            "@version": 1.1,
            "resources": {
                "@id": "http://memory.org/resources",
                "@container": "@index",
                "@index": property_uri
            }
        },
        "resources": {}
    }
    
    # Query all triples with this property
    property_uriref = URIRef(property_uri)
    
    for s, p, o in self.default_graph.triples((None, property_uriref, None)):
        subject_uri = str(s)
        
        # Get the value as a string for indexing
        if isinstance(o, URIRef):
            value = str(o)
        elif isinstance(o, Literal):
            value = str(o)
        else:
            continue
        
        # Add to the appropriate index
        if value not in container["resources"]:
            container["resources"][value] = []
            
        # Add a reference to this entity
        container["resources"][value].append({"@id": subject_uri})
    
    return container


In [ ]:
def test_semantic_memory_lod_agent():
    """Test the Semantic Memory system for a Linked Open Data agent"""
    # Initialize our memory system
    memory = SemanticMemory()
    
    print("=== Testing Basic Functionality ===")
    
    # Test 1: Create and retrieve an entity with UUID
    print("\nTest 1: Create and retrieve entity with UUID")
    entity_data = {
        "http://schema.org/name": "Test Entity",
        "http://schema.org/description": "A test entity for our LOD agent"
    }
    
    entity_id = memory.create_entity_with_uuid(entity_data, "http://schema.org/Thing")
    print(f"Created entity with ID: {entity_id}")
    
    # Retrieve the entity
    entity = memory.query_by_id(entity_id)
    print(f"Retrieved entity: {entity is not None}")
    
    # Test 2: URI dereferencing and caching
    print("\nTest 2: URI dereferencing and caching")
    # For testing, we'll use a well-known URI that should be stable
    test_uri = "https://schema.org/"
    
    # First dereference (should fetch from web)
    print("First dereference (should fetch from web)...")
    start_time = datetime.now()
    schema_data = memory.dereference_uri(test_uri)
    first_fetch_time = (datetime.now() - start_time).total_seconds()
    print(f"First fetch took {first_fetch_time:.2f} seconds")
    
    # Second dereference (should use cache)
    print("Second dereference (should use cache)...")
    start_time = datetime.now()
    schema_data_cached = memory.dereference_uri(test_uri)
    second_fetch_time = (datetime.now() - start_time).total_seconds()
    print(f"Second fetch took {second_fetch_time:.2f} seconds")
    print(f"Cache effective: {second_fetch_time < first_fetch_time}")
    
    # Test 3: Context management
    print("\nTest 3: Context management")
    context_uri = "https://schema.org/"
    context = memory.manage_context(context_uri)
    print(f"Managed context: {context is not None}")
    print(f"Context has {len(context) if isinstance(context, dict) else 0} entries")
    
    # Test 4: Type container
    print("\nTest 4: Type container")
    type_container = memory.build_type_container()
    print(f"Type container has {len(type_container['resources'])} types")
    
    # If we've cached schema.org, we should have some types
    if len(type_container['resources']) > 0:
        print(f"Example types: {list(type_container['resources'].keys())[:3]}")
    
    return memory




In [ ]:
test_memory=test_semantic_memory_lod_agent()

=== Testing Basic Functionality ===

Test 1: Create and retrieve entity with UUID
Created entity with ID: urn:uuid:42107ee4-80c3-4311-9627-5380cd9d4b46
Retrieved entity: True

Test 2: URI dereferencing and caching
First dereference (should fetch from web)...
First fetch took 0.11 seconds
Second dereference (should use cache)...
Second fetch took 0.00 seconds
Cache effective: True

Test 3: Context management
Managed context: True
Context has 2 entries

Test 4: Type container
Type container has 1 types
Example types: ['http://schema.org/Thing']


In [ ]:
def test_semantic_memory_lod_agent_updated():
    """Test the Semantic Memory system with correct schema.org context"""
    # Initialize our memory system
    memory = SemanticMemory()
    
    print("=== Testing with Correct Schema.org Context ===")
    
    # Test Context management with correct URI
    print("\nTest: Context management with correct URI")
    context_uri = "https://schema.org/version/latest/schemaorg-current-https.jsonld"
    
    start_time = datetime.now()
    context = memory.manage_context(context_uri)
    fetch_time = (datetime.now() - start_time).total_seconds()
    
    print(f"Fetched context in {fetch_time:.2f} seconds")
    print(f"Managed context: {context is not None}")
    
    if isinstance(context, dict):
        print(f"Context has {len(context)} entries")
        # Print some example context entries
        context_keys = list(context.keys())[:5]
        print(f"Sample context keys: {context_keys}")
    
    return memory, context


In [ ]:
test_memory_updated=test_semantic_memory_lod_agent_updated()

=== Testing with Correct Schema.org Context ===

Test: Context management with correct URI
Fetched context in 0.74 seconds
Managed context: True
Context has 26 entries
Sample context keys: ['brick', 'csvw', 'dc', 'dcam', 'dcat']


## Memory Functions for Agents

In [ ]:
#| export
@patch
def retrieve_relevant_memory(self:SemanticMemory, structured_query, max_results=5):
    """
    Improved function to retrieve relevant information from memory
    
    Args:
        structured_query: Output from llm_query_translator
        max_results: Maximum number of results to return
        
    Returns:
        JSON-LD data relevant to the query
    """
    relevant_entities = []
    
    # Extract components from the structured query
    entity_ids = structured_query.get("entities", [])
    types = structured_query.get("types", []) 
    properties = structured_query.get("properties", [])
    keywords = structured_query.get("keywords", [])
    
    # Clean keywords (remove punctuation)
    import re
    clean_keywords = [re.sub(r'[^\w\s]', '', keyword.lower()) for keyword in keywords]
    
    # 1. First, check for direct entity matches
    for entity_id in entity_ids:
        if entity_id in self.indices["by_id"]:
            relevant_entities.append(self.indices["by_id"][entity_id])
    
    # 2. Then check for type matches
    for type_uri in types:
        if type_uri in self.indices["by_type"]:
            entities = self.indices["by_type"][type_uri]
            for entity in entities:
                if entity not in relevant_entities:
                    relevant_entities.append(entity)
                    if len(relevant_entities) >= max_results:
                        break
    
    # 3. If we still need more results, try keyword matching
    if len(relevant_entities) < max_results:
        # First, check entity IDs for keyword matches
        for entity_id, entity in self.indices["by_id"].items():
            for keyword in clean_keywords:
                if keyword in entity_id.lower():
                    if entity not in relevant_entities:
                        relevant_entities.append(entity)
                        if len(relevant_entities) >= max_results:
                            break
        
        # Then check entity content for keyword matches
        if len(relevant_entities) < max_results:
            for entity_id, entity in self.indices["by_id"].items():
                entity_str = str(entity).lower()
                for keyword in clean_keywords:
                    if keyword in entity_str:
                        if entity not in relevant_entities:
                            relevant_entities.append(entity)
                            if len(relevant_entities) >= max_results:
                                break
    
    # Compact the results for better readability
    context = {
        "@context": {
            "@vocab": "http://schema.org/",
            "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
            "rdfs": "http://www.w3.org/2000/01/rdf-schema#"
        }
    }
    
    compacted_results = []
    for entity in relevant_entities:
        try:
            compacted = jsonld.compact(entity, context)
            compacted_results.append(compacted)
        except:
            compacted_results.append(entity)
    
    return compacted_results


In [ ]:
#| export
@patch
def query_translator(self:SemanticMemory, natural_language_query):
    """
    Translate a natural language query into structured memory queries
    
    Args:
        natural_language_query: A natural language question or request
        
    Returns:
        Dictionary of possible structured queries
    """
    # This would use an LLM call to translate the query
    # For now, we'll create a simplified version
    
    structured_queries = {
        "by_type": [],
        "by_property": [],
        "keywords": []
    }
    
    # Simple keyword extraction (in a real system, this would use an LLM)
    keywords = [word.lower() for word in natural_language_query.split() 
                if len(word) > 3 and word.lower() not in 
                ['what', 'when', 'where', 'which', 'who', 'how', 'does', 'about']]
    
    structured_queries["keywords"] = keywords
    
    # Check if any keywords match our known types
    for keyword in keywords:
        for type_uri in self.indices["by_type"].keys():
            if keyword in type_uri.lower():
                structured_queries["by_type"].append(type_uri)
    
    return structured_queries


## Testing Agent Memory with Claudette

In [ ]:
model = models[1]
model

'claude-3-7-sonnet-20250219'

In [ ]:
#| export
@patch
def llm_query_translator(self:SemanticMemory, natural_language_query, api_key=None):
    """Use Claude to translate a natural language query into structured memory queries"""
    from claudette import Chat
    
    # Initialize Claude chat
    chat = Chat(model)
    
    # Create a prompt that explains the task
    prompt = f"""
    You are an AI assistant helping to translate natural language queries into structured queries for a semantic memory system.
    
    The memory system contains JSON-LD data with the following types and properties:
    {list(self.indices["by_type"].keys())}
    
    Please convert the following query into a structured JSON format with these fields:
    - entities: List of specific entities mentioned by name or ID
    - types: List of entity types to search for
    - properties: List of properties that are relevant to the query
    - keywords: Important terms from the query
    
    Query: {natural_language_query}
    
    Return only the JSON object, formatted as a Python dictionary.
    """
    
    # Get Claude's response
    response = chat(prompt)
    
    # Extract the structured query from Claude's response
    # In a real implementation, we'd need to parse the response to extract the Python dict
    # For simplicity, we'll use eval() but in production you'd want a safer approach
    try:
        import re
        # Extract code blocks from response
        code_match = re.search(r'```(?:python)?\s*(.*?)\s*```', response, re.DOTALL)
        if code_match:
            structured_query = eval(code_match.group(1))
        else:
            # Try to extract a dictionary directly
            dict_match = re.search(r'({.*})', response, re.DOTALL)
            if dict_match:
                structured_query = eval(dict_match.group(1))
            else:
                # Fallback to simple keyword extraction
                structured_query = {
                    "entities": [],
                    "types": [],
                    "properties": [],
                    "keywords": [w for w in natural_language_query.split() if len(w) > 3]
                }
    except:
        # Fallback to simple keyword extraction
        structured_query = {
            "entities": [],
            "types": [],
            "properties": [],
            "keywords": [w for w in natural_language_query.split() if len(w) > 3]
        }
    
    return structured_query


In [ ]:
#\ export
@patch
def llm_process_information(self:SemanticMemory, query, retrieved_info, api_key=None):
    """Use Claude to process retrieved information and answer a query"""
    from claudette import Chat
    
    # Initialize Claude chat
    chat = Chat(model)
    
    # Format the retrieved information as JSON string
    info_json = json.dumps(retrieved_info, indent=2)
    
    # Create a prompt that explains the task
    prompt = f"""
    You are an AI assistant helping to answer questions based on information retrieved from a semantic memory system.
    
    User query: {query}
    
    Here is the information retrieved from memory:
    ```json
    {info_json}
    ```
    
    Please analyze this information and answer the user's query based ONLY on the information provided.
    If the information is insufficient to answer the query, please state that clearly.
    """
    
    # Get Claude's response
    response = chat(prompt)
    
    return response


In [ ]:
#| export
def test_claude_semantic_memory_improved():
    """Test the Claude-powered semantic memory system with improved retrieval"""
    # Initialize the memory system
    memory = SemanticMemory()
    
    # Seed the memory with sample data
    print("=== Seeding Memory with Sample Data ===")
    
    # Add some example entities
    sample_data = [
        {
            "@context": {"@vocab": "http://schema.org/"},
            "@type": "Person",
            "@id": "http://example.org/person/alice",
            "name": "Alice Smith",
            "jobTitle": "Data Scientist",
            "knows": {"@id": "http://example.org/person/bob"}
        },
        {
            "@context": {"@vocab": "http://schema.org/"},
            "@type": "Person",
            "@id": "http://example.org/person/bob",
            "name": "Bob Johnson",
            "jobTitle": "Software Engineer",
            "knows": {"@id": "http://example.org/person/alice"}
        },
        {
            "@context": {"@vocab": "http://schema.org/"},
            "@type": "Organization",
            "@id": "http://example.org/org/techcorp",
            "name": "TechCorp",
            "employee": [
                {"@id": "http://example.org/person/alice"},
                {"@id": "http://example.org/person/bob"}
            ]
        }
    ]
    
    for data in sample_data:
        memory.add_jsonld(data)
    
    print("Memory seeded with example entities")
    
    # Print memory indices for debugging
    print("\n=== Memory Indices ===")
    print(f"by_id keys: {list(memory.indices['by_id'].keys())}")
    print(f"by_type keys: {list(memory.indices['by_type'].keys())}")
    
    # Test query 1
    user_query = "Who works at TechCorp?"
    print(f"\n=== Processing Query: '{user_query}' ===")
    
    # Step 1: Use Claude to translate the query
    print("\nStep 1: Using Claude to translate the query")
    structured_query = memory.llm_query_translator(user_query)
    print(f"Structured query: {structured_query}")
    
    # Step 2: Retrieve relevant information from memory using improved method
    print("\nStep 2: Retrieving relevant information with improved method")
    relevant_info = memory.retrieve_relevant_memory(structured_query)
    print(f"Found {len(relevant_info)} relevant entities")
    
    # Print the first entity if available
    if relevant_info:
        print("\nFirst relevant entity:")
        print(json.dumps(relevant_info[0], indent=2)[:300] + "...")
    
    # Step 3: Use Claude to process the information and answer
    print("\nStep 3: Using Claude to process information and answer")
    answer = memory.llm_process_information(user_query, relevant_info)
    print(f"Claude's answer: {answer}")
    
    # Test query 2
    user_query2 = "What is Alice's job title?"
    print(f"\n=== Processing Query: '{user_query2}' ===")
    
    # Repeat the process for the second query
    structured_query2 = memory.llm_query_translator(user_query2)
    print(f"Structured query: {structured_query2}")
    
    relevant_info2 = memory.retrieve_relevant_memory(structured_query2)
    print(f"Found {len(relevant_info2)} relevant entities")
    
    if relevant_info2:
        print("\nFirst relevant entity:")
        print(json.dumps(relevant_info2[0], indent=2)[:300] + "...")
    
    answer2 = memory.llm_process_information(user_query2, relevant_info2)
    print(f"Claude's answer: {answer2}")
    
    return memory




In [ ]:
# Run the improved test
test_claude_semantic_memory_improved()

=== Seeding Memory with Sample Data ===
Memory seeded with example entities

=== Memory Indices ===
by_id keys: ['http://example.org/person/alice', 'http://example.org/person/bob', 'http://example.org/org/techcorp']
by_type keys: ['http://schema.org/Person', 'http://schema.org/Organization']

=== Processing Query: 'Who works at TechCorp?' ===

Step 1: Using Claude to translate the query
Structured query: {'entities': [], 'types': [], 'properties': [], 'keywords': ['works', 'TechCorp?']}

Step 2: Retrieving relevant information with improved method
Found 1 relevant entities

First relevant entity:
{
  "@context": {
    "@vocab": "http://schema.org/",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "rdfs": "http://www.w3.org/2000/01/rdf-schema#"
  },
  "@id": "http://example.org/org/techcorp",
  "@type": "Organization",
  "employee": [
    {
      "@id": "http://example.org/perso...

Step 3: Using Claude to process information and answer
Claude's answer: Message(id='msg_011

<__main__.SemanticMemory>

## Use scoped contexts

To implement scoped contexts in our semantic memory system, we'd need to make several key enhancements. Here's a conceptual approach without code:

### 1. Context Preservation

First, we need to modify how we store JSON-LD data to preserve the original contexts:

- Instead of immediately expanding all JSON-LD documents (which loses context information), we should store both the original form and the expanded form
- Track the relationship between contexts and the data they apply to
- Maintain a registry of all contexts encountered, including scoped contexts

### 2. Context-Aware Indexing

Our indexing system needs to become context-aware:

- When indexing properties, record which context they came from
- For each entity, maintain a mapping of which properties belong to which contexts
- Create specialized indices for quickly finding data within specific contexts
- Index terms across contexts but maintain their contextual origin

### 3. Context-Sensitive Retrieval

Enhance retrieval functions to handle scoped contexts:

- Allow queries to specify which context(s) to search within
- When retrieving data, include the relevant context information
- Provide options to view data through different contextual lenses
- Support "context hopping" to follow relationships across context boundaries

### 4. Context Management Tools

Create tools for the agent to work with contexts:

- A tool to list all available contexts in the memory
- A tool to examine a specific context's term definitions
- A tool to translate terms between contexts
- A tool to merge information while preserving contextual boundaries

### 5. Reasoning Across Contexts

Enable the agent to reason across different contexts:

- Implement equivalence mappings between terms in different contexts
- Create inference rules that can operate across contextual boundaries
- Support queries that need to aggregate information from multiple contexts
- Provide mechanisms to resolve conflicts when contexts have contradictory information

### 6. Implementation Approach

For practical implementation, we would:

1. Extend our `SemanticMemory` class with a new `contexts_registry` to store all contexts
2. Modify `add_jsonld` to preserve the original context structure
3. Create specialized data structures for tracking which properties use which contexts
4. Update our indexing to be context-aware
5. Enhance our query tools to support context-specific operations
6. Add visualization tools to help understand the context structure

In [ ]:
#| export
@patch
def _register_contexts(self:SemanticMemory, data, parent_context_id=None):
    """Register all contexts in a JSON-LD document, including scoped contexts"""
    if not isinstance(data, dict):
        return None
    
    # Handle the main context
    if "@context" in data:
        context = data["@context"]
        context_id = self._get_context_id(context)
        
        # If this context is already registered, just return its ID
        if context_id in self.context_registry:
            # Update parent relationship if this is a scoped context
            if parent_context_id:
                self.context_registry[context_id]["parent"] = parent_context_id
            return context_id
        
        # Register the new context
        self.context_registry[context_id] = {
            "context": context,
            "source": "document",  # Or could be URI if it's a remote context
            "parent": parent_context_id,
            "scoped_contexts": {},
            "used_by": set()
        }
        
        # Look for scoped contexts within this context
        if isinstance(context, dict):
            for term, value in context.items():
                if isinstance(value, dict) and "@context" in value:
                    # This term has a scoped context
                    scoped_context_id = self._register_contexts(value, context_id)
                    self.context_registry[context_id]["scoped_contexts"][term] = scoped_context_id
        
        return context_id
    
    return None

In [ ]:
#| export
@patch
def retrieve_in_context(self:SemanticMemory, entity_id, include_scoped_contexts=True):
    """
    Retrieve an entity with its original context structure
    
    Args:
        entity_id: The ID of the entity to retrieve
        include_scoped_contexts: Whether to include scoped contexts in the result
        
    Returns:
        The entity with its original context, or None if not found
    """
    # Check if we have the original data
    if entity_id not in self.original_data:
        return None
    
    # Get the original data
    data = self.original_data[entity_id]
    
    # If scoped contexts are not requested, we can return the data as is
    if not include_scoped_contexts:
        return data
    
    # If we need to handle scoped contexts, we need to do more work
    # First, check if this entity uses contexts with scoped contexts
    if entity_id in self.entity_contexts:
        entity_context = self.entity_contexts[entity_id]
        primary_context_id = entity_context["primary_context"]
        
        if primary_context_id and primary_context_id in self.context_registry:
            context_info = self.context_registry[primary_context_id]
            
            # If this context has scoped contexts, we need to ensure they're included
            if context_info["scoped_contexts"]:
                # Make a deep copy to avoid modifying the original
                import copy
                result = copy.deepcopy(data)
                
                # Ensure the context is a dictionary
                if isinstance(result.get("@context"), str):
                    result["@context"] = {"@vocab": result["@context"]}
                elif isinstance(result.get("@context"), list):
                    # Convert list to dictionary for easier manipulation
                    context_dict = {}
                    for ctx in result["@context"]:
                        if isinstance(ctx, dict):
                            context_dict.update(ctx)
                        elif isinstance(ctx, str):
                            context_dict["@vocab"] = ctx
                    result["@context"] = context_dict
                
                # Add scoped contexts
                for term, scoped_context_id in context_info["scoped_contexts"].items():
                    if scoped_context_id in self.context_registry:
                        scoped_context = self.context_registry[scoped_context_id]["context"]
                        # Ensure the term definition exists
                        if term not in result["@context"]:
                            result["@context"][term] = {}
                        elif not isinstance(result["@context"][term], dict):
                            result["@context"][term] = {"@id": result["@context"][term]}
                        
                        # Add the scoped context
                        result["@context"][term]["@context"] = scoped_context
                
                return result
    
    # If no special handling is needed, return the original data
    return data


In [ ]:
#| export
@patch
def search_with_contexts(self:SemanticMemory, query, context_id=None, max_results=5):
    """
    Search for entities matching a query, preserving context information
    
    Args:
        query: Text to search for
        context_id: Optional context ID to limit search to
        max_results: Maximum number of results to return
        
    Returns:
        List of entities with their original contexts
    """
    # First, find matching entities using our existing search methods
    # but adapted to respect context boundaries if specified
    matching_entities = []
    
    # If a specific context is provided, only search within that context
    if context_id:
        # Get all entities using this context
        if context_id in self.context_registry:
            context_entities = self.context_registry[context_id]["used_by"]
            
            # Search within these entities
            for entity_id in context_entities:
                if entity_id in self.indices["by_id"]:
                    entity = self.indices["by_id"][entity_id]
                    # Simple text search in the entity
                    entity_str = str(entity).lower()
                    if query.lower() in entity_str:
                        matching_entities.append(entity_id)
                        if len(matching_entities) >= max_results:
                            break
    else:
        # Search across all entities
        # First try label-based search
        if "by_label" in self.indices:
            for label, entities in self.indices["by_label"].items():
                if query.lower() in label.lower():
                    for entity in entities:
                        if "@id" in entity and entity["@id"] not in matching_entities:
                            matching_entities.append(entity["@id"])
                            if len(matching_entities) >= max_results:
                                break
        
        # If we need more results, try full text search
        if len(matching_entities) < max_results:
            for entity_id, entity in self.indices["by_id"].items():
                if entity_id not in matching_entities:
                    entity_str = str(entity).lower()
                    if query.lower() in entity_str:
                        matching_entities.append(entity_id)
                        if len(matching_entities) >= max_results:
                            break
    
    # Now retrieve each entity with its context
    results = []
    for entity_id in matching_entities:
        entity_with_context = self.retrieve_in_context(entity_id)
        if entity_with_context:
            results.append(entity_with_context)
    
    return results


In [ ]:
#| export
@patch
def list_contexts(self:SemanticMemory):
    """
    List all available contexts in the memory system
    
    Returns:
        Dictionary with information about all contexts
    """
    result = {}
    
    for context_id, context_info in self.context_registry.items():
        result[context_id] = {
            "parent": context_info["parent"],
            "has_scoped_contexts": len(context_info["scoped_contexts"]) > 0,
            "scoped_context_terms": list(context_info["scoped_contexts"].keys()),
            "entity_count": len(context_info["used_by"])
        }
    
    return result


In [ ]:
#| export
def test_context_aware_memory():
    """Test the context-aware memory system with scoped contexts"""
    # Initialize the memory system
    memory = SemanticMemory()
    
    print("=== Testing Context-Aware Memory System ===")
    
    # Create sample data with scoped contexts
    print("\n1. Creating sample data with scoped contexts")
    
    # Main context for a person
    person_context = {
        "@version": 1.1,
        "@vocab": "http://schema.org/",
        "interests": {
            "@id": "http://xmlns.com/foaf/0.1/interest",
            "@context": {
                "@vocab": "http://xmlns.com/foaf/0.1/",
                "topic": "topic_interest"
            }
        }
    }
    
    # Person data with scoped context for interests
    alice_data = {
        "@context": person_context,
        "@id": "http://example.org/person/alice",
        "@type": "Person",
        "name": "Alice Smith",
        "jobTitle": "Data Scientist",
        "interests": {
            "@id": "http://example.org/interest/semanticweb",
            "name": "Semantic Web",
            "topic": "Linked Data"  # Uses the scoped context
        }
    }
    
    # Add the data to memory
    memory.add_jsonld(alice_data)
    
    print("\n2. Checking context registry")
    contexts = memory.list_contexts()
    print(f"Found {len(contexts)} contexts in registry")
    for context_id, info in contexts.items():
        print(f"Context: {context_id}")
        print(f"  Has scoped contexts: {info['has_scoped_contexts']}")
        if info['has_scoped_contexts']:
            print(f"  Scoped context terms: {info['scoped_context_terms']}")
        print(f"  Used by {info['entity_count']} entities")
    
    print("\n3. Retrieving entity with context")
    entity = memory.retrieve_in_context("http://example.org/person/alice")
    
    if entity:
        print("Successfully retrieved entity with context")
        
        # Check if the scoped context is preserved
        if "@context" in entity and isinstance(entity["@context"], dict):
            if "interests" in entity["@context"] and isinstance(entity["@context"]["interests"], dict):
                if "@context" in entity["@context"]["interests"]:
                    print("Scoped context for 'interests' is preserved!")
                    print(f"Scoped context: {entity['@context']['interests']['@context']}")
                else:
                    print("Scoped context for 'interests' is missing!")
            else:
                print("'interests' term is not properly defined in context")
        else:
            print("Entity context is not properly preserved")
    else:
        print("Failed to retrieve entity")
    
    print("\n4. Testing search with contexts")
    search_results = memory.search_with_contexts("Semantic Web")
    print(f"Search found {len(search_results)} results")
    
    if search_results:
        print("First search result:")
        if "@context" in search_results[0] and "interests" in search_results[0]:
            print(f"Entity: {search_results[0]['@id']}")
            print(f"Has interests: {search_results[0]['interests']['name']}")
            if "topic" in search_results[0]["interests"]:
                print(f"Interest topic: {search_results[0]['interests']['topic']}")
    
    # Create another entity with a different context structure
    print("\n5. Adding entity with different context")
    bob_data = {
        "@context": {
            "@vocab": "http://schema.org/",
            "knows": {"@type": "@id"}
        },
        "@id": "http://example.org/person/bob",
        "@type": "Person",
        "name": "Bob Johnson",
        "knows": "http://example.org/person/alice"
    }
    
    memory.add_jsonld(bob_data)
    
    # Test cross-context search
    print("\n6. Testing cross-context search")
    cross_results = memory.search_with_contexts("Johnson")
    print(f"Cross-context search found {len(cross_results)} results")
    
    if cross_results:
        print("Cross-context search result:")
        print(f"Entity: {cross_results[0]['@id']}")
        print(f"Name: {cross_results[0]['name']}")
        
        # Check if this entity knows Alice
        if "knows" in cross_results[0]:
            print(f"Knows: {cross_results[0]['knows']}")
    
    return memory




In [ ]:
test_context_aware_memory()

=== Testing Context-Aware Memory System ===

1. Creating sample data with scoped contexts

2. Checking context registry
Found 2 contexts in registry
Context: context:a04031139a9d4ff036c9a3d2048dc43b
  Has scoped contexts: True
  Scoped context terms: ['interests']
  Used by 1 entities
Context: context:ee5d178f115bc151af1717cd98a341aa
  Has scoped contexts: False
  Used by 0 entities

3. Retrieving entity with context
Successfully retrieved entity with context
Scoped context for 'interests' is preserved!
Scoped context: {'@vocab': 'http://xmlns.com/foaf/0.1/', 'topic': 'topic_interest'}

4. Testing search with contexts
Search found 1 results
First search result:
Entity: http://example.org/person/alice
Has interests: Semantic Web
Interest topic: Linked Data

5. Adding entity with different context

6. Testing cross-context search
Cross-context search found 1 results
Cross-context search result:
Entity: http://example.org/person/bob
Name: Bob Johnson
Knows: http://example.org/person/alice

<__main__.SemanticMemory>

## Test Claude Integration of Semantic Memory with Scoped Contexts

In [ ]:
#| export
@patch
def answer_query_with_context_aware_tools(self:SemanticMemory, user_query):
    """Use Claude with context-aware tools to answer queries about the memory store"""
    from claudette import Chat
    
    # Define context-aware tools as properly annotated functions
    def list_available_contexts() -> dict:
        """List all available contexts in the memory store with their details"""
        return self.list_contexts()
    
    def search_across_contexts(query: str) -> list:
        """Search for entities matching a query across all contexts"""
        results = self.search_with_contexts(query)
        return results
    
    def search_in_context(query: str, context_id: str) -> list:
        """Search for entities matching a query within a specific context"""
        results = self.search_with_contexts(query, context_id)
        return results
    
    def get_entity_with_context(entity_id: str) -> dict:
        """Get an entity with its original context structure"""
        return self.retrieve_in_context(entity_id)
    
    def examine_context(context_id: str) -> dict:
        """Examine a specific context and its structure"""
        if context_id in self.context_registry:
            context_info = self.context_registry[context_id]
            return {
                "context": context_info["context"],
                "parent": context_info["parent"],
                "scoped_contexts": context_info["scoped_contexts"],
                "entity_count": len(context_info["used_by"]),
                "entities": list(context_info["used_by"])
            }
        return {"error": f"Context {context_id} not found"}
    
    # Create chat with our context-aware tools
    chat = Chat(model, tools=[
        list_available_contexts,
        search_across_contexts,
        search_in_context,
        get_entity_with_context,
        examine_context
    ])
    
    # Create a prompt that explains the task and the importance of contexts
    prompt = f"""
    You are an AI assistant with access to a semantic memory store that preserves context information.
    
    In JSON-LD, contexts define how terms map to IRIs and how data should be interpreted. 
    A scoped context applies only to specific parts of a document, allowing different vocabularies 
    to be used in different sections.
    
    Use the available tools to explore the memory store and answer the following query:
    
    {user_query}
    
    When analyzing data, pay attention to contexts and scoped contexts. Different properties 
    might have different meanings depending on their context. Always preserve context 
    information when presenting results.
    """
    
    # Use the toolloop method to let Claude use multiple tools
    result = chat.toolloop(prompt)
    return result


In [ ]:
def test_claude_context_aware_memory():
    """Test Claude integration with context-aware memory"""
    # Initialize and populate the memory system
    memory = SemanticMemory()
    
    print("=== Testing Claude Integration with Context-Aware Memory ===")
    
    # Create sample data with scoped contexts
    print("\n1. Creating sample data with multiple contexts")
    
    # Academic context for publications
    academic_context = {
        "@version": 1.1,
        "@vocab": "http://schema.org/",
        "keywords": "keywords",
        "references": {
            "@id": "citation",
            "@context": {
                "@vocab": "http://purl.org/dc/terms/",
                "title": "title",
                "year": "date"
            }
        }
    }
    
    # Social context for people
    social_context = {
        "@version": 1.1,
        "@vocab": "http://xmlns.com/foaf/0.1/",
        "name": "name",
        "knows": {"@type": "@id"},
        "interests": {
            "@id": "interest",
            "@context": {
                "@vocab": "http://dbpedia.org/ontology/",
                "topic": "subject"
            }
        }
    }
    
    # Add a publication with academic context
    paper_data = {
        "@context": academic_context,
        "@id": "http://example.org/publication/semantic-web",
        "@type": "ScholarlyArticle",
        "name": "Understanding the Semantic Web",
        "author": "Alice Smith",
        "keywords": ["Linked Data", "RDF", "JSON-LD"],
        "references": [
            {
                "@id": "http://example.org/publication/json-ld",
                "title": "JSON-LD 1.1",
                "year": "2020"
            }
        ]
    }
    
    # Add a person with social context
    person_data = {
        "@context": social_context,
        "@id": "http://example.org/person/alice",
        "@type": "Person",
        "name": "Alice Smith",
        "knows": "http://example.org/person/bob",
        "interests": [
            {
                "@id": "http://dbpedia.org/resource/Semantic_Web",
                "name": "Semantic Web",
                "topic": "Computer Science"
            }
        ]
    }
    
    # Add another person
    person2_data = {
        "@context": social_context,
        "@id": "http://example.org/person/bob",
        "@type": "Person",
        "name": "Bob Johnson",
        "knows": "http://example.org/person/alice",
        "interests": [
            {
                "@id": "http://dbpedia.org/resource/Artificial_Intelligence",
                "name": "Artificial Intelligence",
                "topic": "Computer Science"
            }
        ]
    }
    
    memory.add_jsonld(paper_data)
    memory.add_jsonld(person_data)
    memory.add_jsonld(person2_data)
    
    print("Memory populated with sample data")
    
    # Test queries that require context awareness
    queries = [
        "What are Alice's interests and how do they relate to her publications?",
        "How are interests represented differently from references in our data?",
        "What contexts are available in the memory system and what do they contain?"
    ]
    
    for i, query in enumerate(queries):
        print(f"\n=== Query {i+1}: {query} ===")
        response = memory.answer_query_with_context_aware_tools(query)
        print(f"\nClaude's response:\n{response}")
    
    return memory



In [ ]:
# Run the test
claude_test_memory = test_claude_context_aware_memory()


=== Testing Claude Integration with Context-Aware Memory ===

1. Creating sample data with multiple contexts
Memory populated with sample data

=== Query 1: What are Alice's interests and how do they relate to her publications? ===

Claude's response:
Message(id='msg_01C8xXdoV6ic3wwoA3R8mofY', content=[TextBlock(citations=None, text='Based on the information retrieved from the memory store, I can now answer your question about Alice\'s interests and how they relate to her publications:\n\n## Alice\'s Interests and Their Relation to Her Publications\n\n### Alice\'s Interests\nAccording to the data in the FOAF (Friend of a Friend) context, Alice Smith has the following interest:\n- **Semantic Web** (identified as `http://dbpedia.org/resource/Semantic_Web`)\n- This interest is categorized under the topic of **Computer Science** (using the DBpedia ontology context)\n\n### Alice\'s Publications\nAlice is the author of a scholarly article titled **"Understanding the Semantic Web"** (identifi

In [ ]:
#| export
def test_supply_chain_example():
    """Test the context-aware memory system with a supply chain example"""
    # Initialize the memory system
    memory = SemanticMemory()
    
    print("=== Supply Chain Example with Context-Aware Memory ===")
    
    # Create contexts for different parts of the supply chain
    
    # Manufacturing context
    manufacturing_context = {
        "@version": 1.1,
        "@vocab": "http://example.org/manufacturing/",
        "components": {
            "@container": "@list",
            "@context": {
                "@vocab": "http://example.org/components/",
                "material": "materialType",
                "supplier": {"@type": "@id"}
            }
        },
        "assembledAt": "productionFacility",
        "qualityScore": {"@type": "http://www.w3.org/2001/XMLSchema#decimal"}
    }
    
    # Logistics context
    logistics_context = {
        "@version": 1.1,
        "@vocab": "http://example.org/logistics/",
        "shipment": {
            "@context": {
                "@vocab": "http://example.org/shipping/",
                "status": "currentStatus",
                "location": "currentLocation",
                "estimatedArrival": {"@type": "http://www.w3.org/2001/XMLSchema#dateTime"}
            }
        },
        "trackingNumber": "shipmentIdentifier"
    }
    
    # Retail context
    retail_context = {
        "@version": 1.1,
        "@vocab": "http://schema.org/",
        "inventory": {
            "@context": {
                "@vocab": "http://example.org/inventory/",
                "level": "stockLevel",
                "reorderPoint": "minimumStockLevel"
            }
        },
        "price": {"@type": "http://schema.org/PriceSpecification"},
        "category": "category"
    }
    
    # Supplier context
    supplier_context = {
        "@version": 1.1,
        "@vocab": "http://example.org/supplier/",
        "certification": {
            "@context": {
                "@vocab": "http://example.org/certification/",
                "type": "certificationType",
                "validUntil": {"@type": "http://www.w3.org/2001/XMLSchema#date"}
            }
        },
        "contactPerson": "representativeName"
    }
    
    print("\n1. Creating supply chain entities with different contexts")
    
    # Create a product in the manufacturing context
    smartphone_product = {
        "@context": manufacturing_context,
        "@id": "http://example.org/product/smartphone-x200",
        "@type": "Product",
        "productName": "Smartphone X200",
        "productionDate": "2023-10-15",
        "batchNumber": "SX200-2023-10-B42",
        "assembledAt": "Shenzhen Factory 7",
        "qualityScore": "9.8",
        "components": [
            {
                "@id": "http://example.org/component/screen-amoled",
                "name": "AMOLED Display",
                "material": "Glass-Composite",
                "partNumber": "SCR-X200-AM",
                "supplier": "http://example.org/supplier/displaytech"
            },
            {
                "@id": "http://example.org/component/battery-lithium",
                "name": "Lithium Battery Pack",
                "material": "Lithium-Ion",
                "partNumber": "BAT-5000mAh",
                "supplier": "http://example.org/supplier/powerplus"
            },
            {
                "@id": "http://example.org/component/processor-a12",
                "name": "A12 Processor",
                "material": "Silicon",
                "partNumber": "CPU-A12-3.1",
                "supplier": "http://example.org/supplier/chipworks"
            }
        ]
    }
    
    # Create a shipment in the logistics context
    smartphone_shipment = {
        "@context": logistics_context,
        "@id": "http://example.org/shipment/sx200-batch-42",
        "@type": "Shipment",
        "shipmentDate": "2023-10-20",
        "contents": "Smartphone X200 - 1000 units",
        "origin": "Shenzhen Factory 7",
        "destination": "North American Distribution Center",
        "trackingNumber": "SHP-2023-10-42-NAM",
        "carrier": "Global Logistics Partners",
        "shipment": {
            "status": "In Transit",
            "location": "Pacific Ocean",
            "estimatedArrival": "2023-11-05T08:00:00Z",
            "mode": "Sea Freight",
            "container": "GLPC-98765"
        }
    }
    
    # Create a retail listing in the retail context
    smartphone_retail = {
        "@context": retail_context,
        "@id": "http://example.org/retail/smartphone-x200",
        "@type": "Product",
        "name": "Smartphone X200",
        "description": "Latest flagship smartphone with A12 processor and AMOLED display",
        "price": {
            "@type": "PriceSpecification",
            "price": "899.99",
            "priceCurrency": "USD"
        },
        "category": "Electronics/Smartphones",
        "brand": "TechX",
        "sku": "TX-SX200-BLK",
        "inventory": {
            "level": "750",
            "reorderPoint": "150",
            "warehouse": "North American Distribution Center",
            "lastUpdated": "2023-11-10"
        }
    }
    
    # Create supplier information in the supplier context
    display_supplier = {
        "@context": supplier_context,
        "@id": "http://example.org/supplier/displaytech",
        "@type": "Supplier",
        "name": "DisplayTech Industries",
        "location": "Seoul, South Korea",
        "establishedYear": "1998",
        "contactPerson": "Ji-Hun Park",
        "email": "contracts@displaytech.kr",
        "preferredSupplier": "true",
        "certification": [
            {
                "type": "ISO 9001",
                "issuedBy": "International Standards Organization",
                "validUntil": "2025-06-30"
            },
            {
                "type": "Environmental Management",
                "issuedBy": "Green Electronics Council",
                "validUntil": "2024-12-31"
            }
        ]
    }
    
    # Add all entities to memory
    memory.add_jsonld(smartphone_product)
    memory.add_jsonld(smartphone_shipment)
    memory.add_jsonld(smartphone_retail)
    memory.add_jsonld(display_supplier)
    
    print("Supply chain entities added to memory")
    
    # Print context registry
    contexts = memory.list_contexts()
    print(f"\n2. Context registry contains {len(contexts)} contexts")
    
    # Test queries that require understanding across contexts
    queries = [
        "What is the current status of the Smartphone X200 product across the supply chain?",
        "How does the supplier certification information relate to the components in the smartphone?",
        "What is the journey of the Smartphone X200 from manufacturing to retail?"
    ]
    
    for i, query in enumerate(queries):
        print(f"\n=== Query {i+1}: {query} ===")
        response = memory.answer_query_with_context_aware_tools(query)
        print(f"\nClaude's response:\n{response}")
    
    return memory

In [ ]:
test_supply_chain_example()

=== Supply Chain Example with Context-Aware Memory ===

1. Creating supply chain entities with different contexts
Supply chain entities added to memory

2. Context registry contains 8 contexts

=== Query 1: What is the current status of the Smartphone X200 product across the supply chain? ===

Claude's response:
Message(id='msg_01N5KihP87K7m3dpXqBo2TGb', content=[TextBlock(citations=None, text="Based on the information gathered from the memory store, here's the current status of the Smartphone X200 product across the supply chain:\n\n## Smartphone X200 Supply Chain Status\n\n### Manufacturing Information\n- **Product Name**: Smartphone X200\n- **Production Date**: October 15, 2023\n- **Batch Number**: SX200-2023-10-B42\n- **Assembly Location**: Shenzhen Factory 7\n- **Quality Score**: 9.8 (on a scale of 10)\n\n### Components (with context-specific information)\n1. **AMOLED Display**\n   - Material: Glass-Composite\n   - Part Number: SCR-X200-AM\n   - Supplier: DisplayTech Industries (S

<__main__.SemanticMemory>

In [ ]:
#| export
@patch
def add_jsonld_with_graph(self:SemanticMemory, data, context=None):
    """Enhanced version of add_jsonld that properly handles @graph structures while preserving contexts.
    
    Args:
        data: JSON-LD data to add
        context: Optional context to apply
        
    Returns:
        The expanded data
    """
    # If data contains a @graph, process each entity in the graph
    if isinstance(data, dict) and '@graph' in data:
        # Extract the context if present
        graph_context = data.get('@context')
        
        # Register the graph context if it exists
        if graph_context:
            graph_context_id = self._register_contexts({"@context": graph_context})
        
        # Process each entity in the graph
        results = []
        for entity in data['@graph']:
            # Apply the graph context to each entity if not already present
            if graph_context and '@context' not in entity:
                entity_with_context = entity.copy()
                entity_with_context['@context'] = graph_context
                result = self.add_jsonld(entity_with_context, context)
            else:
                result = self.add_jsonld(entity, context)
            results.append(result)
        
        return results
    
    # If no @graph, use the context-aware method
    return self.add_jsonld(data, context)


In [ ]:
#| export
@patch
def add_named_graph(self:SemanticMemory, graph_data, graph_id):
    """Add a named graph to the memory system, preserving context information."""
    # Ensure graph_data has a @graph property
    if not isinstance(graph_data, dict):
        graph_data = {"@graph": graph_data}
    elif '@graph' not in graph_data:
        graph_data = {"@graph": [graph_data]}
    
    # Add the graph ID
    if graph_id:
        graph_data['@id'] = graph_id
    
    # Process the graph
    result = self.add_jsonld_with_graph(graph_data)
    
    # Get or create the named graph in the RDF dataset
    named_graph = self.dataset.graph(URIRef(graph_id))
    
    # Convert the expanded data to RDF and add to the named graph
    # We need to reprocess the data to get it into the named graph
    for entity in graph_data.get('@graph', []):
        # Create a temporary graph
        temp_graph = Graph()
        
        # Add the context to the entity if not present
        if '@context' not in entity and '@context' in graph_data:
            entity = entity.copy()
            entity['@context'] = graph_data['@context']
        
        # Parse the entity into the temporary graph
        temp_graph.parse(data=json.dumps(entity), format='json-ld')
        
        # Add all triples from the temporary graph to the named graph
        for s, p, o in temp_graph:
            named_graph.add((s, p, o))
    
    # Add metadata about this graph to the default graph
    if graph_id:
        self.default_graph.add((URIRef(graph_id), RDF.type, URIRef("http://www.w3.org/ns/prov#Entity")))
        self.default_graph.add((URIRef(graph_id), 
                               URIRef("http://www.w3.org/ns/prov#generatedAtTime"), 
                               Literal(datetime.now().isoformat())))
    
    return result


In [ ]:
#| export
@patch
def query_named_graph(self:SemanticMemory, graph_id, query_text):
    """
    Query a specific named graph for entities matching the query text
    
    Args:
        graph_id: ID of the named graph to query
        query_text: Text to search for in the graph
        
    Returns:
        List of matching entities with their properties
    """
    named_graph = self.dataset.graph(URIRef(graph_id))
    
    # Find entities matching the query text
    matching_entities = []
    
    # First check literals for matches
    for s, p, o in named_graph:
        if isinstance(o, Literal) and query_text.lower() in str(o).lower():
            if str(s) not in matching_entities:
                matching_entities.append(str(s))
    
    # Collect all properties for matching entities
    results = []
    for entity_uri in matching_entities:
        entity = {"@id": entity_uri}
        
        for s, p, o in named_graph.triples((URIRef(entity_uri), None, None)):
            pred = str(p)
            if pred not in entity:
                entity[pred] = []
            
            if isinstance(o, URIRef):
                entity[pred].append({"@id": str(o)})
            elif isinstance(o, Literal):
                entity[pred].append({"@value": str(o)})
        
        results.append(entity)
    
    return results


In [ ]:
#| export
@patch
def list_named_graphs(self:SemanticMemory):
    """
    List all named graphs in the memory system
    
    Returns:
        Dictionary of graph IDs and their metadata
    """
    graphs = {}
    
    # Get all graph identifiers
    for graph_identifier in self.dataset.graphs():
        graph_id = str(graph_identifier)
        if graph_id == "urn:x-arq:DefaultGraph":
            continue  # Skip default graph
            
        # Get graph
        graph = self.dataset.graph(graph_identifier)
        
        # Count triples
        triple_count = len(list(graph))
        
        # Get entities (subjects)
        entities = set()
        for s in graph.subjects():
            entities.add(str(s))
            
        # Get metadata about the graph from the default graph
        metadata = {}
        for p, o in self.default_graph.predicate_objects(graph_identifier):
            pred = str(p)
            if isinstance(o, URIRef):
                metadata[pred] = str(o)
            elif isinstance(o, Literal):
                metadata[pred] = str(o)
        
        # Store graph info
        graphs[graph_id] = {
            "triple_count": triple_count,
            "entity_count": len(entities),
            "metadata": metadata
        }
    
    return graphs


In [ ]:
@patch
def answer_query_with_graph_aware_tools(self:SemanticMemory, user_query):
    """Use Claude with graph-aware tools to answer queries about the memory store"""
    from claudette import Chat
    
    # Define graph-aware tools with proper type annotations
    def list_available_contexts() -> dict:
        """List all available contexts in the memory store with their details"""
        return self.list_contexts()
    
    def list_available_graphs() -> dict:
        """List all named graphs in the memory store"""
        return self.list_named_graphs()
    
    def search_across_contexts(query: str) -> list:
        """Search for entities matching a query across all contexts"""
        return self.search_with_contexts(query)
    
    def search_in_named_graph(graph_id: str, query: str) -> list:
        """Search for entities matching a query within a specific named graph"""
        return self.query_named_graph(graph_id, query)
    
    def get_entity_with_context(entity_id: str) -> dict:
        """Get an entity with its original context structure"""
        return self.retrieve_in_context(entity_id)
    
    # Create chat with our graph-aware tools
    chat = Chat(model, tools=[
        list_available_contexts,
        list_available_graphs,
        search_across_contexts,
        search_in_named_graph,
        get_entity_with_context
    ])
    
    # Create a prompt that explains the task
    prompt = f"""
    You are an AI assistant with access to a semantic memory store that preserves context information 
    and supports named graphs.
    
    Named graphs allow for organizing information into separate logical groups. Each named graph 
    can have its own context and contain different types of information.
    
    Use the available tools to explore the memory store and answer the following query:
    
    {user_query}
    
    When analyzing data, pay attention to which named graph the information comes from, as this 
    provides important context about the domain and purpose of the information.
    """
    
    # Use the toolloop method to let Claude use multiple tools
    result = chat.toolloop(prompt)
    return result


In [ ]:
def test_graph_aware_system():
    """Test the complete graph-aware semantic memory system"""
    memory = SemanticMemory()
    
    print("=== Testing Graph-Aware Semantic Memory System ===")
    
    # Add people data to one graph
    people_graph = {
        "@context": {
            "@version": 1.1,
            "@vocab": "http://xmlns.com/foaf/0.1/",
            "name": "name",
            "knows": {"@id": "knows", "@type": "@id"}
        },
        "@graph": [
            {
                "@id": "http://example.org/person/alice",
                "@type": "Person",
                "name": "Alice Smith",
                "knows": ["http://example.org/person/bob", "http://example.org/person/charlie"]
            },
            {
                "@id": "http://example.org/person/bob",
                "@type": "Person",
                "name": "Bob Johnson",
                "knows": "http://example.org/person/alice"
            },
            {
                "@id": "http://example.org/person/charlie",
                "@type": "Person",
                "name": "Charlie Brown",
                "knows": "http://example.org/person/alice"
            }
        ]
    }
    
    # Add organization data to another graph
    org_graph = {
        "@context": {
            "@version": 1.1,
            "@vocab": "http://schema.org/",
            "name": "name",
            "member": {"@id": "member", "@type": "@id"}
        },
        "@graph": [
            {
                "@id": "http://example.org/org/techcorp",
                "@type": "Organization",
                "name": "TechCorp",
                "member": ["http://example.org/person/alice", "http://example.org/person/bob"]
            },
            {
                "@id": "http://example.org/org/researchlab",
                "@type": "ResearchOrganization",
                "name": "Research Lab",
                "member": "http://example.org/person/charlie"
            }
        ]
    }
    
    # Add project data to a third graph
    project_graph = {
        "@context": {
            "@version": 1.1,
            "@vocab": "http://example.org/project/",
            "name": "name",
            "contributor": {"@id": "contributor", "@type": "@id"},
            "organization": {"@id": "organization", "@type": "@id"}
        },
        "@graph": [
            {
                "@id": "http://example.org/project/semantic-web",
                "@type": "Project",
                "name": "Semantic Web Initiative",
                "contributor": ["http://example.org/person/alice", "http://example.org/person/charlie"],
                "organization": "http://example.org/org/researchlab"
            },
            {
                "@id": "http://example.org/project/knowledge-graph",
                "@type": "Project",
                "name": "Enterprise Knowledge Graph",
                "contributor": "http://example.org/person/bob",
                "organization": "http://example.org/org/techcorp"
            }
        ]
    }
    
    # Add the graphs to memory
    print("\n1. Adding multiple named graphs")
    memory.add_named_graph(people_graph, "http://example.org/graphs/people")
    memory.add_named_graph(org_graph, "http://example.org/graphs/organizations")
    memory.add_named_graph(project_graph, "http://example.org/graphs/projects")
    
    # List all graphs
    print("\n2. Listing all named graphs")
    graphs = memory.list_named_graphs()
    for graph_id, info in graphs.items():
        print(f"  Graph: {graph_id}")
        print(f"    Triples: {info['triple_count']}")
        print(f"    Entities: {info['entity_count']}")
    
    # Test cross-graph query
    print("\n3. Testing cross-graph query")
    query = "What projects is Alice contributing to and which organizations are involved?"
    print(f"Query: '{query}'")
    response = memory.answer_query_with_graph_aware_tools(query)
    print(f"Claude's response:\n{response}")
    
    return memory


In [ ]:
test_graph_aware_system()

=== Testing Graph-Aware Semantic Memory System ===

1. Adding multiple named graphs

2. Listing all named graphs
  Graph: <http://example.org/graphs/people> a rdfg:Graph;rdflib:storage [a rdflib:Store;rdfs:label 'Memory'].
    Triples: 10
    Entities: 3
  Graph: <urn:x-arq:DefaultGraph> a rdfg:Graph;rdflib:storage [a rdflib:Store;rdfs:label 'Memory'].
    Triples: 32
    Entities: 10
  Graph: <http://example.org/graphs/organizations> a rdfg:Graph;rdflib:storage [a rdflib:Store;rdfs:label 'Memory'].
    Triples: 7
    Entities: 2
  Graph: <http://example.org/graphs/projects> a rdfg:Graph;rdflib:storage [a rdflib:Store;rdfs:label 'Memory'].
    Triples: 9
    Entities: 2
  Graph: <urn:x-rdflib:default> a rdfg:Graph;rdflib:storage [a rdflib:Store;rdfs:label 'Memory'].
    Triples: 0
    Entities: 0

3. Testing cross-graph query
Query: 'What projects is Alice contributing to and which organizations are involved?'
Claude's response:
Message(id='msg_018V91uhTRKtYbUU1wdz82Uo', content=[TextB

<__main__.SemanticMemory>

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()